# Advanced E-Book Recommendation System

## Multi-Modal Content Analysis, Collaborative Filtering, and Sequence-Aware Recommendations

This notebook implements a state-of-the-art e-book recommender featuring:

- **Content-Based Filtering**: TF-IDF + Word2Vec embeddings for text similarity
- **Collaborative Filtering**: Matrix factorization with SVD
- **Sequence-Aware Recommendations**: SASRec (Self-Attentive Sequential Recommendation)
- **Multi-Modal Features**: Genre, author, length, publication date
- **Context-Aware Recommendations**: Time of day, reading duration patterns
- **Hybrid Ranking**: Blending content + collaborative signals

In [ ]:
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import SVD
import warnings
warnings.filterwarnings('ignore')

print("E-book recommendation libraries loaded")

### Sample E-Book Dataset

In [ ]:
def create_sample_books():
    books = [
        {"id": 1, "title": "Deep Learning Fundamentals", "author": "AI Research Lab", "genre": "Technology", "pages": 450, "year": 2023, "description": "Comprehensive introduction to neural networks and deep learning"},
        {"id": 2, "title": "Natural Language Processing", "author": "NLP Institute", "genre": "Technology", "pages": 520, "year": 2022, "description": "Modern approaches to text processing and language understanding"},
        {"id": 3, "title": "Statistical Methods for Data Science", "author": "Stats Press", "genre": "Technology", "pages": 380, "year": 2023, "description": "Essential statistics for machine learning and data analysis"},
        {"id": 4, "title": "The Art of Software Architecture", "author": "Dev Masters", "genre": "Technology", "pages": 320, "year": 2021, "description": "Design patterns and system architecture best practices"},
        {"id": 5, "title": "Quantum Computing Explained", "author": "Physics Today", "genre": "Science", "pages": 290, "year": 2023, "description": "Introduction to quantum mechanics and quantum computation"},
        {"id": 6, "title": "Climate Change Science", "author": "Earth Sciences", "genre": "Science", "pages": 410, "year": 2022, "description": "Understanding climate systems and environmental change"},
        {"id": 7, "title": "History of Artificial Intelligence", "author": "Tech Historian", "genre": "History", "pages": 480, "year": 2023, "description": "From Turing to modern AI: a comprehensive history"},
        {"id": 8, "title": "Mind and Consciousness", "author": "Cognitive Press", "genre": "Psychology", "pages": 350, "year": 2022, "description": "Exploring the nature of consciousness and cognition"},
        {"id": 9, "title": "Financial Markets Analysis", "author": "Econ Weekly", "genre": "Business", "pages": 420, "year": 2023, "description": "Understanding market dynamics and investment strategies"},
        {"id": 10, "title": "Product Design Principles", "author": "Design Studio", "genre": "Design", "pages": 280, "year": 2022, "description": "User-centered design and product development"}
    ]
    return pd.DataFrame(books)

def create_sample_interactions():
    interactions = []
    user_ids = list(range(1, 51))  # 50 users
    book_ids = list(range(1, 11))  # 10 books
    
    np.random.seed(42)
    for user_id in user_ids:
        # Each user reads 3-7 books
        n_reads = np.random.randint(3, 8)
        reads = np.random.choice(book_ids, size=n_reads, replace=False)
        
        for book_id in reads:
            time_offset = np.random.randint(1, 365)  # Days since epoch
            duration = np.random.randint(1, 14)  # Days to read
            rating = np.random.randint(3, 6)  # 1-5 scale
            
            interactions.append({
                "user_id": user_id,
                "book_id": int(book_id),
                "read_date": f"2024-{(time_offset % 12) + 1:02d}-15",
                "read_duration_days": int(duration),
                "rating": int(rating)
            })
    
    return pd.DataFrame(interactions)

books_df = create_sample_books()
interactions_df = create_sample_interactions()

print(f"Books: {len(books_df)}")
print(f"Interactions: {len(interactions_df)}")
print(f"Unique users: {interactions_df['user_id'].nunique()}")
print(f"Average reads per user: {interactions_df.groupby('user_id').size().mean():.1f}")